# Data Preprocessing Strategy
**Goal:** Refactored Data Preprocessing Pipeline with Raw Signal Segmentation.

In [ ]:
import os
import sys
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import random
from typing import List, Dict, Tuple
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view

sys.path.append("../src")
from CrossDomainFeatureExtractor import CrossDomainFeatureExtractor
from ConstructGroundTruthUsingCUSUM import UnivariateCUSUMDetector

INPUT_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\Downsampled"
OUTPUT_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\LSTM_Inputs"
os.makedirs(OUTPUT_PATH, exist_ok=True)

WINDOW_SIZES = [10, 20, 30]
XJTU_SAMPLING_RATE = 25600.0
MAX_FREQ_HZ = 1280.0

SEGMENT_LENGTH = 1024
STRIDE = 512

In [ ]:
def calculate_series_health_index(t: int, t_cp: int, t_f: int) -> float:
    """
    Calculate the health index.
    BHI(t) = 1 - [ max(t - Tcp, 0) / (Tf - Tcp) ]
    Enforces HI[-1] = 0.0 limit.
    """
    if t_f <= t_cp: return 0.0
    if t <= t_cp: return 1.0
    hi = 1.0 - (max(t - t_cp, 0) / (t_f - t_cp))
    return max(0.0, hi)

def segment_and_extract(file_path: str, bearing_id: str) -> Tuple[pd.DataFrame, Dict]:
    df_raw = pd.read_pickle(file_path)
    extractor = CrossDomainFeatureExtractor(sampling_rate=XJTU_SAMPLING_RATE, max_freq_hz=MAX_FREQ_HZ)
    features_list = []
    
    # 1. Raw Signal Segmentation (Phase 1)
    for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc=f"Ext. {bearing_id}", leave=False):
        try:
            h_sig = np.array(row['H_Signal'], dtype=float)
            if len(h_sig) < SEGMENT_LENGTH: continue
            segments = sliding_window_view(h_sig, window_shape=SEGMENT_LENGTH)[::STRIDE]
            for seg in segments:
                feats = extractor.extract_all_features(seg)
                feats['Minute'] = row['Minute']
                feats['Bearing_ID'] = bearing_id
                features_list.append(feats)
        except Exception:
            continue
            
    df_features = pd.DataFrame(features_list)
    if df_features.empty: return df_features, {}
    feat_cols = [c for c in df_features.columns if c not in ['Minute', 'Bearing_ID']]
    
    # 2. CUSUM Target Estimation
    detector = UnivariateCUSUMDetector(baseline_ratio=0.1, k_factor=0.5, h_factor=5.0)
    rms_col = "td_rms" if "td_rms" in df_features.columns else feat_cols[0]
    change_point, _ = detector.fit_predict(df_features[rms_col].values)
    
    # 3. HI and RUL Construction (Phase 2)
    n_total = len(df_features)
    t_f = n_total - 1
    health_indices = []
    rul_labels = []
    for t in range(n_total):
        hi = calculate_series_health_index(t, change_point, t_f)
        health_indices.append(hi)
        max_rul = t_f - change_point
        rul = max(0, t_f - t)
        rul_labels.append(min(rul, max_rul))
            
    df_features['Change_Point'] = change_point
    df_features['Health_Index'] = health_indices
    df_features['Target_RUL'] = rul_labels

    # Enforce Hard Constraints
    df_features.loc[df_features.index[-1], 'Health_Index'] = 0.0
    df_features.loc[df_features.index[-1], 'Target_RUL'] = 0.0

    # 4. Negative Transfer Alignment
    for col in feat_cols:
        if df_features[col].corr(df_features['Health_Index']) > 0:
            df_features[col] = df_features[col] * -1.0

    # 5. Global MinMax Scaling (Fix Feature Explosion)
    scaler = MinMaxScaler()
    df_features[feat_cols] = scaler.fit_transform(df_features[feat_cols])

    return df_features, {"Bearing_ID": bearing_id, "Total_Lifespan": n_total, "Change_Point": change_point}

def create_sliding_windows(data_df: pd.DataFrame, window_size: int) -> pd.DataFrame:
    """Uses sliding_window_view for Chronological Sequence Windows (Phase 3)."""
    if len(data_df) <= window_size: return pd.DataFrame()
    features_col = [c for c in data_df.columns if c not in ["Change_Point", "Health_Index", "Target_RUL", "Minute", "Bearing_ID", "Normalized_Bearing_ID"]]
    X_data = data_df[features_col].values
    X_windows = sliding_window_view(X_data, window_shape=window_size, axis=0) # shape: (N - W + 1, F, W) 
    
    n_windows = X_windows.shape[0]
    out_rows = []
    # Target alignment: y is the last time step of the window
    targets_hi = data_df['Health_Index'].values[window_size - 1:]
    targets_rul = data_df['Target_RUL'].values[window_size - 1:]
    b_id = data_df['Bearing_ID'].values[-1]
    
    for i in range(n_windows):
        row_record = {}
        for f_idx, feat in enumerate(features_col):
            for t in range(window_size):
                row_record[f"{feat}_t{t}"] = X_windows[i, f_idx, t]
        row_record['Health_Index'] = targets_hi[i]
        row_record['Target_RUL'] = targets_rul[i]
        row_record['Bearing_ID'] = b_id
        out_rows.append(row_record)
    return pd.DataFrame(out_rows)

def get_stratified_folds() -> List[Dict[str, List[str]]]:
    folds = [
        {"val": ["Bearing1_1", "Bearing2_1", "Bearing3_1"]},
        {"val": ["Bearing1_2", "Bearing2_2", "Bearing3_2"]},
        {"val": ["Bearing1_3", "Bearing2_3", "Bearing3_3"]},
        {"val": ["Bearing1_4", "Bearing2_4", "Bearing3_4"]},
        {"val": ["Bearing1_5", "Bearing2_5", "Bearing3_5"]}
    ]
    all_bearings = [f"Bearing{c}_{i}" for c in range(1, 4) for i in range(1, 6)]
    for fold in folds:
        fold["train"] = [b for b in all_bearings if b not in fold["val"]]
    return folds

def preprocess_and_slide():
    import glob
    aggregated_files = glob.glob(os.path.join(INPUT_PATH, "*", "*_aggregated.pkl"))
    if not aggregated_files: return
        
    all_bearings_features = {}
    all_metadata = []
    
    for file in tqdm(aggregated_files, desc="Processing Bearings"):
        raw_b_id = os.path.basename(file).replace("_aggregated.pkl", "")
        parent_dir = os.path.basename(os.path.dirname(file))
        c_idx = 1
        if '37.5' in parent_dir: c_idx = 2
        elif '40' in parent_dir: c_idx = 3
        b_num = raw_b_id.split("_")[-1] if raw_b_id.startswith("Bearing") else "1"
        bearing_id = f"Bearing{c_idx}_{b_num}"
        
        df_feat, metadata = segment_and_extract(file, bearing_id)
        if not df_feat.empty:
            df_feat['Normalized_Bearing_ID'] = bearing_id
            all_bearings_features[bearing_id] = df_feat
            all_metadata.append(metadata)

    pd.DataFrame(all_metadata).to_csv(os.path.join(OUTPUT_PATH, "bearing_metadata.csv"), index=False)
    folds = get_stratified_folds()
    
    for w_size in WINDOW_SIZES:
        W_DIR = os.path.join(OUTPUT_PATH, f"window_size_{w_size}")
        os.makedirs(W_DIR, exist_ok=True)
        windowed_dfs = {b_id: create_sliding_windows(df_feat, w_size) for b_id, df_feat in all_bearings_features.items()}
        for fold_idx, cv_split in enumerate(folds, start=1):
            val_list = [windowed_dfs[b] for b in cv_split['val'] if b in windowed_dfs and not windowed_dfs[b].empty]
            if val_list:
                pd.concat(val_list, ignore_index=True).to_csv(os.path.join(W_DIR, f"processed_val_fold{fold_idx}.csv"), index=False)
            train_list = [windowed_dfs[b] for b in cv_split['train'] if b in windowed_dfs and not windowed_dfs[b].empty]
            if train_list:
                pd.concat(train_list, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True).to_csv(os.path.join(W_DIR, f"processed_train_fold{fold_idx}.csv"), index=False)

if __name__ == "__main__":
    preprocess_and_slide()